# 3.3 SES Prototype — Signal Loss (Reproducible GitHub Version)
**Purpose:** Train a lightweight anomaly detector for Signal Loss, generate the dashboard artefacts, and save them into the existing repository structure.

**Inputs (repo):** `data/processed/X_test_sample.csv` (sample features)

**Outputs (repo):**
- `reports/figures/signal_loss_event_shap_values.csv`
- `reports/figures/signal_loss_event_heatmap.png`
- `reports/figures/signal_loss_continuous_heatmap.png`
- `data/processed/test_scores_raw.csv`
- `data/processed/sl_test_scores.csv`
- `data/processed/sl_test_eventized_scores.csv`

Notes:
- This notebook is designed to be run from `notebooks/` in the GitHub repo.
- The modelling choices here are intentionally lightweight to support reproducibility and artefact generation for the Streamlit dashboard.

In [ ]:
# ============================================================
# 0) Imports
# ============================================================
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestRegressor

import matplotlib.pyplot as plt

# Optional: SHAP (required to generate the CSV heatmap)
import shap


In [ ]:
# ============================================================
# 1) Resolve repo paths (DO NOT change repo tree)
# ============================================================
HERE = Path.cwd().resolve()
REPO = HERE
while not (REPO / "app.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent

assert (REPO / "app.py").exists(), f"Repo root not found. Current working dir: {HERE}"

DATA_DIR = REPO / "data" / "processed"
FIG_DIR  = REPO / "reports" / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

IN_X = DATA_DIR / "X_test_sample.csv"

print("REPO    :", REPO)
print("DATA_DIR:", DATA_DIR)
print("FIG_DIR :", FIG_DIR)
print("IN_X    :", IN_X)


In [ ]:
# ============================================================
# 2) Load input features
# ============================================================
if not IN_X.exists():
    raise FileNotFoundError(
        f"Missing input file: {IN_X}.\n"
        "Expected: data/processed/X_test_sample.csv"
    )

df = pd.read_csv(IN_X)

# Heuristics: if a timestamp column exists, use it; otherwise create one
time_candidates = [c for c in df.columns if c.lower() in ("time", "timestamp", "datetime", "date")]
if time_candidates:
    tcol = time_candidates[0]
    df[tcol] = pd.to_datetime(df[tcol], errors="coerce", utc=True)
else:
    tcol = "timestamp"
    # create a synthetic regular timeline (1-minute spacing) for reproducible plots
    df[tcol] = pd.date_range("2021-11-01", periods=len(df), freq="min", tz="UTC")

# Feature columns: exclude obvious non-features
drop_cols = {tcol}
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(method="ffill").fillna(0)

print("Rows:", len(df))
print("Features:", len(feature_cols))
print("Time column:", tcol)


In [ ]:
# ============================================================
# 3) Train an unsupervised detector and compute anomaly scores
#    (lightweight, deterministic, reproducible)
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values)

iso = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)
iso.fit(X_scaled)

# IsolationForest: higher = more normal; invert and min-max scale to [0,1] as "anomaly_score"
raw_score = -iso.decision_function(X_scaled)
raw_score = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min() + 1e-12)

scores = pd.DataFrame({
    "timestamp": pd.to_datetime(df[tcol], utc=True),
    "anomaly_score": raw_score,
    "proba_raw": raw_score,  # dashboard sometimes expects proba_raw naming
})

# Persist dashboard inputs (match app.py paths)
OUT_TEST_SCORES_RAW = DATA_DIR / "test_scores_raw.csv"
OUT_SL_TEST_SCORES  = DATA_DIR / "sl_test_scores.csv"

scores.to_csv(OUT_TEST_SCORES_RAW, index=False)
scores.rename(columns={"timestamp": "time"}).to_csv(OUT_SL_TEST_SCORES, index=False)

print("Saved:", OUT_TEST_SCORES_RAW)
print("Saved:", OUT_SL_TEST_SCORES)


In [ ]:
# ============================================================
# 4) Eventization (create sl_test_eventized_scores.csv)
# ============================================================
# Conservative: mark top 1% windows as anomalies
thr = float(np.quantile(scores["anomaly_score"].values, 0.99))
scores["is_anomaly"] = scores["anomaly_score"] >= thr

scores = scores.sort_values("timestamp").reset_index(drop=True)
scores["grp"] = (scores["is_anomaly"] != scores["is_anomaly"].shift(1)).cumsum()

events = []
for g, chunk in scores.groupby("grp"):
    if not bool(chunk["is_anomaly"].iloc[0]):
        continue
    t_start = chunk["timestamp"].iloc[0]
    t_end   = chunk["timestamp"].iloc[-1]
    events.append({
        "t_start": t_start,
        "t_end": t_end,
        "label": 1,
        "severity": "high",
    })

events_df = pd.DataFrame(events)
OUT_EVENTS = DATA_DIR / "sl_test_eventized_scores.csv"
events_df.to_csv(OUT_EVENTS, index=False)

print("Threshold (99th percentile):", thr)
print("Events:", len(events_df))
print("Saved:", OUT_EVENTS)


## SHAP artefacts for the dashboard
The Streamlit dashboard loads the **event SHAP matrix CSV** from:
`reports/figures/signal_loss_event_shap_values.csv`.

It will also display heatmap PNGs if present:
- `reports/figures/signal_loss_event_heatmap.png`
- `reports/figures/signal_loss_continuous_heatmap.png`

In [ ]:
# ============================================================
# 5) SHAP helper: save matrix CSV in the format expected by app.py
#    index = features, columns = time steps
# ============================================================
def save_shap_matrix_csv(out_csv: Path, shap_matrix_2d: np.ndarray, feature_names: list[str], time_labels: list[str] | None = None) -> None:
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    arr = np.asarray(shap_matrix_2d, dtype=float)

    if arr.ndim != 2:
        raise ValueError(f"Expected 2D matrix. Got shape={arr.shape}")

    if arr.shape[0] != len(feature_names):
        raise ValueError(
            f"Shape mismatch: matrix has {arr.shape[0]} rows but {len(feature_names)} features."
        )

    if time_labels is None:
        time_labels = [f"t{i}" for i in range(arr.shape[1])]

    df_out = pd.DataFrame(arr, index=feature_names, columns=time_labels)
    df_out.to_csv(out_csv)
    print(f"Saved SHAP matrix CSV -> {out_csv}")


In [ ]:
# ============================================================
# 6) Train a surrogate regressor and compute SHAP values
#    (explains anomaly_score in terms of feature contributions)
# ============================================================
# Surrogate model for SHAP (fast + stable)
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_scaled, scores["anomaly_score"].values)

explainer = shap.TreeExplainer(rf)

# Pick an "event window" around the highest anomaly score (or the first event if available)
if not events_df.empty:
    # window around first event
    t0 = pd.to_datetime(events_df["t_start"].iloc[0], utc=True)
    t1 = pd.to_datetime(events_df["t_end"].iloc[0], utc=True)
    # include some padding
    pad = pd.Timedelta(minutes=10)
    w_start, w_end = t0 - pad, t1 + pad
    mask = (scores["timestamp"] >= w_start) & (scores["timestamp"] <= w_end)
    idxs = np.where(mask.values)[0]
else:
    # fallback: top-scoring point ±10 minutes
    peak_idx = int(np.argmax(scores["anomaly_score"].values))
    idxs = np.arange(max(0, peak_idx - 10), min(len(scores), peak_idx + 11))

# Reduce to a manageable number of timesteps for heatmap readability
max_steps = 30
if len(idxs) > max_steps:
    # sample evenly
    idxs = np.linspace(idxs.min(), idxs.max(), max_steps).round().astype(int)

X_win = X_scaled[idxs]  # shape: (time_steps, n_features)

# SHAP values: shape (time_steps, n_features)
shap_vals = explainer.shap_values(X_win)
shap_vals = np.asarray(shap_vals)

# Convert to (features, time_steps) for dashboard heatmap
shap_matrix = shap_vals.T  # (n_features, time_steps)
time_labels = [f"t{i}" for i in range(shap_matrix.shape[1])]

OUT_SHAP_CSV = FIG_DIR / "signal_loss_event_shap_values.csv"
save_shap_matrix_csv(OUT_SHAP_CSV, shap_matrix, feature_cols, time_labels=time_labels)


In [ ]:
# ============================================================
# 7) Save event heatmap PNG (matplotlib)
# ============================================================
OUT_EVENT_PNG = FIG_DIR / "signal_loss_event_heatmap.png"

plt.figure(figsize=(12, 6))
plt.imshow(shap_matrix, aspect="auto")
plt.yticks(np.arange(len(feature_cols)), feature_cols, fontsize=7)
plt.xticks(np.arange(len(time_labels)), time_labels, rotation=90, fontsize=7)
plt.title("Signal Loss – SHAP heatmap around one detected high-score window (surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_EVENT_PNG, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", OUT_EVENT_PNG)


In [ ]:
# ============================================================
# 8) Continuous heatmap PNG (overview)
#    Compute SHAP for a subsample of windows and aggregate by time order.
# ============================================================
OUT_CONT_PNG = FIG_DIR / "signal_loss_continuous_heatmap.png"

# Subsample windows for efficiency / readability
rng = np.random.default_rng(42)
n_windows = min(120, len(X_scaled))
sel = rng.choice(len(X_scaled), size=n_windows, replace=False)
sel = np.sort(sel)

X_sub = X_scaled[sel]
shap_sub = np.asarray(explainer.shap_values(X_sub))  # (n_windows, n_features)

# Use mean absolute SHAP to pick top features for display
mean_abs = np.mean(np.abs(shap_sub), axis=0)
top_n = min(18, len(feature_cols))
top_idx = np.argsort(mean_abs)[::-1][:top_n]

# Build matrix (features x time)
cont_matrix = shap_sub[:, top_idx].T  # (top_features, n_windows)
cont_feat_names = [feature_cols[i] for i in top_idx]
cont_time_labels = [f"w{i}" for i in range(n_windows)]

plt.figure(figsize=(12, 6))
plt.imshow(cont_matrix, aspect="auto")
plt.yticks(np.arange(len(cont_feat_names)), cont_feat_names, fontsize=7)
plt.xticks(np.arange(len(cont_time_labels))[::10], cont_time_labels[::10], rotation=90, fontsize=7)
plt.title("Signal Loss – Continuous SHAP overview (top features; surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_CONT_PNG, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", OUT_CONT_PNG)


## Completion check
At this point the Streamlit dashboard should be able to render:
- the SHAP matrix heatmap from the CSV, and
- the time-series trend from `data/processed/test_scores_raw.csv`.

If the dashboard still falls back to PNGs, verify that the files exist at the paths printed above.